# Experimentation Part II - Data Preprocessing and Simple Analyses


[Cosimo Iaia](https://fiebachlab.org/team/iaia), Christian Fiebach



Dear all, welcome back after the Christmas break, and thank you for handing in your experiment scripts and csv-datasets!!!

In this final session of the course, we will work with these data and try to read the into python to have a first look at them. 



## Outline

Within this notebook we will go through a few steps that frequently occur when getting our data. These include:

1. Building a dataset 
- Reading in the log files from our subjects  
- Having a look at the data
- Identifying potential problems and cleaning them
- Defining which variables to include into the dataset
- Writing the variables to a new dataset
2. Visualizing the data  
3. Calculating descriptive statistics
4. Info on final exam (Modulpruefung)

## Building Our Dataset

Note that we can expect to do a bit of pre-processing ... because a quick glance already showed that there is a bit of variation between some of the files. So some of the first things we may have to do is to `clean` files a bit, so that they can be read in automatically and converted into a `dataset` with which we can work efficiently. 

<cr>
    
*(Even though probably different from the current case, this kind of cleaning or pre-processing is not so uncommon in experimental work. Some cases over which we may stumble in "everyday lab life" might be missing triggers in an EEG dataset, so that we have to reconstruct timing and add it into a file post hoc, or correct a systematic delay that occurred due to technical problems in some subjects, etc.)*

### Reading in the log files

The first thing we have to do is access our experimental log files which contain the data. Remember that you all submitted a *.csv file, so to work with those (e.g., loop over them, open them ...), we need a `list` of these *.csv files. We will need quite flexible code here, as we did not define strict filenames conventions. Thus, filenames could very quite a bit across files. 

In [ ]:
import pandas as pd
from os import listdir

In [ ]:
#Define your directory where you store your data
cwd = '/Users/cfiebach/Desktop/pfp25_psychopy_data/'

#List of all files contained in your directory
files = listdir(cwd)

#Here, make sure that you have only csv files: 
#i.e., exclude any file that is not a csv
files = [file for file in files if '.csv' in file] # this is a list comprehension


#Note that this is equivalent to the following code: 

#files_csv = list()

#for file in files: 
    
#    if '.csv' in file: 
        
#        file_csv.append(file)

#   ... just that this would generate a new list (files_csv) from files, which only contains *.csv files!


As a test, let's print the list and see if it contains the twelve files we expect it to contain:

In [ ]:
## You all know how to print a list, right?
## Please try out after the course if not ...
## Hint: What would be an elegant way to do this? Do you remember the enumerate function?

OK, what next? 

### Dataset

Maybe we first have a look at our data - we have only a few short datasets - so we can actually look into them to check if everything is ok. For this, we want to `loop` over the filenames, open them (i.e., read them in) and maybe first check if all individual datasets have the same length. 

In [ ]:
#Initilize an empty list in which you store the files you open
dfs = list()

#Loop through the file names ...
for sub_id, file in enumerate(files): 
    
    # ... and open the current file in pandas
    df = pd.read_csv(cwd + file) # the default separator here is (sep = ',')

    #Let's use the .column attribute combine with len() to check for the number of columns in each file:
    n_columns = list(df.columns)
    
    #This will print the number of columns:
    print(file, f' columns found: {len(n_columns)}')
    

OK, so there are quite obvious differences here that we need to fix. Maybe we have a quick look at the datasets themselves, right? We can use almost the same syntax for this ...

In [ ]:
#Initilize an empty list in which you store the files you open
dfs = list()

#dictionary for remapping the name of the columns
#this is needed because not all files are in the same format/naming scheme
#rename_columns = {'reaction_time': 'rating_rt', 
#                  'item': 'stimulus'}

#we will only extract what we need
columns_to_include = ['stimulus', 'rating', 'rating_rt']

#loop through the file names
for sub_id, file in enumerate(files): 
    
    #open in pandas
    df = pd.read_csv(cwd + file) # the default separator here is (sep = ',')
    print(df)
    
   


Did you notice the problem?


Yes, indeed! Some of the file contents are not `comma separated` but use semicolon instead. This is less standard than comma separation - and actually a problem, because `python assumes comma as a separator` when reading in `*.csv` files. We can see that if this is not the case, python / pandas will will not separate the columns correctly when reading in the datafiles.
We need to fix this!

One approach to fix this prohblem would be to specify a different separator (the semicolon) for these two datasets. 

In [ ]:
#Initilize an empty list in which you store the files you open
dfs = list()

#loop through the file names
for sub_id, file in enumerate(files): 
        
    # ... and open the current file in pandas
    df = pd.read_csv(cwd + file) # the default separator here is (sep = ',')

    #Let's use the .column attribute combine with len() to check for the number of columns in each file:
    n_columns = list(df.columns)
    
    #this statement will account for this
    #we basically assume that if the n of columns is one or less, then the file has semicolon
    if len(n_columns) <=1: 
        
        #reopen it but specify semicolon as separator
        df = pd.read_csv(cwd + file, sep = ';')
        
        #and check the number of columns as above ...
        n_columns = list(df.columns)   
        print(file, f' columns found: {len(n_columns)}')

        #and print the respective dataset
        print(df)



**OK, this looks better now! Seems we have found a little routine to fix this problem!**

There seems to be another problem - did you notice it? Sometimes the variable identifying our items is called `stimulus`, sometimes `item` - we want it to be called stimulus. And we want to use `rating_rt` not `reaction_time` - se we also need to rename these columns in some datasets, before we can merge the individual files into a single dataset. To do so, we use the `rename` attribute of `pandas`. 

Lastly, we want to keep it simple and only extract three variables from the datasets, `stimulus`, `rating`, and `rating_rt`. These shall be included into a new dataset together with a subject identifier, `sub_id`. You have probably noticed that we have generated this variable using the `enumerate` command in the previous cells ...

In [ ]:
#Initilize an empty list in which you store the files you open
dfs = list()

#Define a dictionary for remapping the names of the columns in some datafiles
#this is needed because not all files are in the same format/naming scheme
rename_columns = {'reaction_time': 'rating_rt', 
                  'item': 'stimulus'}

#Define the variables to extract from the datasets
columns_to_include = ['stimulus', 'rating', 'rating_rt']


#loop through the file names
for sub_id, file in enumerate(files): 
    
    #open in pandas
    df = pd.read_csv(cwd + file) # the default separator here is (sep = ',')
    
    #Get the number of columns in each file, to correct the ones separated by semicolon
    n_columns = list(df.columns)
    
    #Print filename and the number of columns
    print(file, f' columns found: {len(n_columns)}')
    
    #Correct files with only 1 column, assuming that these files had semicolon as separator
    if len(n_columns) <=1: 
        
        #reopen it but specify semicolon as separator
        df = pd.read_csv(cwd + file, sep = ';')
        
        #and check the number of columns as above ...
        n_columns = list(df.columns)   
        print(f'  ... corrected! columns now: {len(n_columns)}')

        #NEW: 
        #remap the column names if needed, so that everything is in the same format
        df = df.rename(columns = rename_columns)
        
        #extract only the columns we need
        df = df[columns_to_include]
        
        #add a sub_id to keep track of different subject
        df['sub_id'] = str(sub_id) # sub_id comes from enumerate at the beginning of the loop

    #Proceed for all other datasets, i.e., in case n_columns is actually more then one    
    else: 
        
        #NEW: 
        #remap the column names 
        df = df.rename(columns = rename_columns)
        
        #extract only the the columns we need
        df = df[columns_to_include]
        
        #add a sub_id to keep track of different subject
        df['sub_id'] = str(sub_id)
    
    #In all cases, store the current file in the final dataset by appending it to the list called dfs
    dfs.append(df)
    


Now let's have a look at the list `dfs` we generated. It seems it is not yet the kind of dataframe we want to have ...

In [ ]:
dfs

What is the problem here? It is a list of dataframes. One potential problem here may be, that each index appears multiple time and that different items can have the same index in different subjects. One problem that may occur here, for example, is that when looping through the dataset with `for`, the routine implicitly calls these indices, so that the `for` loop may be in a different order than you expected!

What we want to have, thus, is only one dataframe to work with when doing statistics. For this, we want to concatenate all entries in the `dfs` list into one, and `reset` the indices. Luckily, there is a command that can do just this!

In [ ]:
df = pd.concat(dfs).reset_index(drop = True) 


In [ ]:
df

**This looks right!!!**

## Visualizing Data

The next step one often does is to visualize descriptive statistics of data, to get some feeling for how your data look like. In our case, we could for example look at the `mean ratings` and `mean response times` across participants for each word (as the number of words in our study is not too large ...).

For this, we can use common libraries for plotting, `seaborn` and `matplotlib`. `seaborn` in particular is built to work on top of `pandas`and easy to use. `matplotlib` can work with anything (e.g., lists, numpy arrays) but requires more lines of code. Here, we use a combination of both. 


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt 

#initialize a new figure
plt.figure(figsize= (14, 8))
sns.barplot(data = df, x = 'stimulus', y = 'rating')
plt.tight_layout()



Isn't this super-efficient? We did not even have to specify that we want `seaborn` to calculate the mean - it just did so automatically. When providing a `categorical x variable` (here: stimulus) and a `numerical y variable`, it is the `default` that data are grouped by the x variable and the mean is calculated across all members of each group! You should be able to find out here: https://seaborn.pydata.org/tutorial/categorical.html and here: https://seaborn.pydata.org/tutorial/error_bars.html  how to change this default behavior, e.g., if you want to visualize a different measure of central tendency. 

And what do the error bars show? Again, the default was used, which is actually **not** the standard deviation, but a confidence interval (specifically: a 95% confidence interval determined by bootstrapping). We can try to use the standard deviation by setting the `errorbar` attribute and compare the plots:

In [ ]:
#initialize a new figure
plt.figure(figsize= (14, 8))
sns.barplot(data = df, x = 'stimulus', y = 'rating', errorbar='sd')
plt.tight_layout()


For the sake of completeness, let's do the same for the time taken to give the ratings, i.e., `rating_rt`. There are many ways to format your figures nicely, including adding titles and subtitles, changing the color of the bars, specifying text for the axis labels, controlling the axes, the tick marks on the axes, font and font size, etc etc. As an example, we could add a bit more color to the next plot and include a title:

In [ ]:
#NOTE: if we would not open a new figure, the plot would be in the same figure you opened before
plt.figure(figsize= (14, 8))
sns.barplot(data = df, x = 'stimulus', y = 'rating_rt', hue = 'stimulus', palette = 'Set2')
plt.tight_layout
plt.title('Mean Response Time per Word')

You can play around with this, for example based on the following website: https://seaborn.pydata.org/tutorial/introduction.html 
Here, you also find a nice overview over different types of beautiful data plots, which could be inspiring for your future work!!!

Another interesting site might be `data to viz` which was suggested by one of you: https://www.data-to-viz.com/

In [ ]:
## you can try out further formatting options of the figure here ... 
## For example, you can try to format the title differently or try out different color palettes.

## Calculating Some Descriptive Statistics

Often we want to calculate some intiail descriptive statistics to get a feeling for our data. Here we show, as an example, how to calculate the `mean` quite efficiently for each word using `pandas`. You can easily adapt this to calculate other statistics like the `.median()` or the standard deviation `.std()`. You will easily find web resources for this.

In [ ]:
## Compute mean ratings and mean RTs per word 

# first define three lists to work with
words = list()
ratings = list()
rts = list()

# you can group by a specific label/condition in pandas in a loop!
# The following loop very efficiently partitions our dataset df into separate dataframes (which we here
# call 'df_rating') for every word stimulus in the dataset. The word (i.e., stimulus column) is used
# to group the dataset, and the 'word specific' dataframe (df_ratings) will contain the variables 
# from our dataset which we want to average, i.e., rating and rating_rt.
# Another way to say this is that df_ratings is a dataframe that contains only the rows with stimulus == word
# If this is difficult to understand, you can use print() commands to 'visualize' what happens on every iteration ...

for word, df_ratings in df.groupby(by = 'stimulus'): 
    
    #keep track of the word for this dataframe
    words.append(word)
    
    #compute mean rating for this word
    mean_rating = df_ratings['rating'].mean()
    ratings.append(mean_rating) #store
    


Let's have a quick look at the sliced dataframe - i.e., one iteration of this loop. 

In [ ]:
df_ratings #this calls the last iteration only

Now we want to build a dataset with the results. This can become helpful, when for example the plan is to continue working with the results calculated here. It also can be a nice way to display the results in a notebook - for the small number of words we used, this actually works in our case.

As the command below shows, **a dataframe (df) is actually built from a dictionary** in `pandas`, where each key becomes a column and each value (which is typically a list or an array) will be the observations or rows in that column. Note that this will work only if all the values have the same dimensions - if you have 20 ratings, you need to have 20 reaction times!

A good idea from a coding perspective is to check whether this precondition for building a dataset is fulfilled, which you can do like this:


In [ ]:
assert len(words) == len(ratings) == len(rts)

#if this assertion is False, you get an assertion error and you need to check your code to 
#find a mistake

OK, there seems to be an error here - we need to fix it!

If you have fixed it, you can now build the new dataframe which we want to call `res`.

In [ ]:
#build df for results 
res = pd.DataFrame({'words': words, 
                    'mean_ratings': ratings, 
                    'mean_rt': rts})

Let's have a lool at the results:

In [ ]:
#calling the new dataset `res` which cotains the calculated mean values per word.
res

Admittedly, this is just another way to get to the same data as in the plotting section above - that also showed the means ... Still, this is a showcase of what you can do with `pandas`!

*If you want to save this dataframe, remember that you saw how to do this during the psychopy session!*

## Calculating a Correlation

The last thing we want to do in today's session is to calculate a `correlation` between the two variables we acquired - the rating and the response time, i.e., the time it took to rate the respective stimulus word. More specifically, we want to compute this correlation based on the mean response times and ratings across subjects, for each word! So we can use the dataset `res` we just generated to calculate that correlation. (Why do we want to look at this correlation? We might have hypothesized that concrete words are easier to recognition or that it might take longer to judge an abstract word on the concrete-abstract dimention. Let's find it out ...)

In the following, we will use two different ways to calculate this correlation, and we will also visualize it.

#### 1. Calculating the correlation using scipy

For this, we use scipy's `pearsonr` correlation, to calculate a Pearson correlation. `scipy.stats` has also `spearmanr` etc. Note that we want to calculate the correlation value itself and test it for significance!

In [ ]:
import numpy as np # always a good idea to import it 
from scipy.stats import pearsonr

# extract the two variables to be correlated ...

x = res['mean_ratings'].values # this is now a numpy array
y = res['mean_rt'].values

# ... and have a look at them
x, y



In [ ]:
# compute the pearson correlation and print the output
pearson_corr, p_value = pearsonr(x, y)
pearson_corr, p_value

OK, there does seem to be a significant correlation of around *r* = .56. This means that more abstract words (with higher ratings) took on average longer to respond to.

#### 2. Calculating the correlation with built-in methods in pandas

In [ ]:

# The following command will return a correlation matrix.
# It computes the corr for all the columns you pass (it can be more than two).
# As default, Pearson correlation is used, but you can specify spearman or kendall by doing .corr(method = 'spearman')
corrs = res[['mean_ratings', 'mean_rt']].corr() 



In [ ]:
#it shoud be the same as scipy.stats
corrs

#### 3. Scatterplot with regression line

To this end, we use the plotting libraries we have imported above!

In [ ]:
#let's plot the scatterplot
plt.figure() #new figure
sns.scatterplot(data = res, x = 'mean_ratings', y = 'mean_rt')
plt.tight_layout()

#plot regression line
# Note that this time we do not open a new figure, because we want to plot the line into the scatterplot we just made! 
sns.regplot(data = res, x = 'mean_ratings', y = 'mean_rt', scatter = False) 
# we set scatter to False because it is already plotted

To save a figure, you would call matplotlib again:

In [ ]:
# You have to specify a path which is the full path (folder + filename).
# Do not forget the file format (png or svg, etc)!

# The command looks like this:
#plt.savefig('folder/name.png') 



Note that this saves the last figure opened! So if you want to save multple figures (like the figures above), you need to add this line of code after you plot something, each time.

<cr>
<cr>    
</cr>
    
## Final Exam

If you choose to do the `Modulpruefung` in this course, you have to complete a final assignment to get a grade!

You have already programmed an experiment and submitted a dataset ... this will count as one part of the exam. And you have all done a great job here!

The second part will be to take the data we worked with today and: 

- test them for the presence of outlier(s) and - if present - exclude those outliers from the remainder of the data analysis, 
- define a new variable that separates the words into two experimental conditions: abstract vs. concrete words. 
- When done with this, use this variable first to plot mean and standard deviation for each condition, then
- compare the ratings and the response times between abstract and concrete words using a t test.
- Last but not least, please report what you did and the results in a jupyter notebook that you hand in as your exam.


#### Author Guide Notebook
The notebook should have a nice structure with headlines, text sections and code and results. You should read in the data, describe how you test for outlier(s) and implement this test, exclude outliers if necessary. Then, you should describe how you generate the condition variable (abstract vs. concrete) and implement this. Following this, please (a) visualizee and describe mean differences between abstract and concrete words (don't forget errorbars!), describe the statistical test and implemnt it and report its results. 
Please also don't forget to include your name and Matrikelnumber into the document when you submit.

#### Testing for Outliers
There are many ways to test for outliers. A frequently used approach is to test whether a subject's dependent values (e.g., ratings, RTs) are beyond 3 standard deviations from the mean of the remaining sample. So you might want to iterate over participants and calculate, on every iteration, the mean and std across all but one participant and compare that participant's mean to the mean of the others.

This would be tested for each dependent variable. If you want to be more sophisticated, you could even test this separately for every condition. 

#### Abstract vs. Concrete Words
It should be quite obvious which are which, but just to make sure: 
Abstract words: Ehre, Frieden, Hoffnung, Klugheit, Liebe, Ruhm, Seele, Tragik, Wahrheit, Weisheit
Concrete words: Anker, Apfel, Bagger, Besen, Felsen, Gabel, Hammer, Pfanne, Stuhl, Toaster

### Comparing Abstract and Concrete Words
To plot a mean for comparing experimental conditions, you would calculate it in a way to first calculate a mean within each participant (e.g., a mean across all concrete words within every participant) and then calculate a mean across participants. This 'grand mean' is the mean value you want to plot.

The first t test (rating) will have more the character of a manipulation check - it would be really weird if abstract words would not be rated as more abstract in our sample ... However, it is not unplausible to assume that these ratings take longer for abstract words. We would like to test this assumption with a t test. 

To calculate the t test, you can use scipy or statsmodels packages. I think we have done a t test before!?!

#### Data 
You can get the data from here: https://github.com/cfiebach/Python_For_Psychologists_25-26/blob/main/lecture/experiments/pfp25_psychopy_data.zip

#### Submission of Assignment
Please e-mail the final jupyter notebook to fiebach[ at ]psych.uni-frankfurt.de.

**Deadline is March 15, 2026!**
